[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c06_barrels.ipynb)

# Kepler's wine barrels: maxima and minima

In 1613, in Linz, a wine merchant priced Johannes Kepler's barrels by pushing one rod through the hole in each
barrel's side, to the far edge of the lid. Kepler worked out that the Austrian barrels sat almost exactly on the
shape that holds the most, where a curve is **flat at its top**. That is the idea behind maxima and minima.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
Run button) to run it. Run each cell in turn, from the top. When a slider appears, drag it and watch. Change any
number you like and run the cell again. Nothing can break. It works in your browser (no install) and in Google
Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this notebook. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the curve · slopes · slices

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

ROD = 1.0                                   # the rod reading, in metres, in every barrel below

def across(h):
    """The lid's diameter when the rod reads ROD: across² + (half the height)² = ROD²."""
    return math.sqrt(max(0.0, ROD ** 2 - (h / 2) ** 2))

def litres(h):
    """A barrel drawn as a cylinder (Kepler's first step): π r² h, in litres."""
    return math.pi * (across(h) / 2) ** 2 * h * 1000

H_BEST = 2 / math.sqrt(3)                   # where the volume curve is flat (worked out in section 3)
print(f"the best barrel: {H_BEST:.3f} m tall, {across(H_BEST) * 100:.1f} cm across, {litres(H_BEST):.0f} litres")

## 1 · One rod reading, many barrels

The merchant's rod goes in at the hole in the middle of the side and reaches the far edge of a lid. Its length is
the long side of a right triangle: the short sides are the width of the barrel and half its height.

So with the reading fixed at one metre, a taller barrel must be narrower. Drag the height and watch how much wine
each shape holds. The curve on the right is the volume for every height.

In [ ]:
HS = np.linspace(0.001, 2 * ROD - 0.001, 400)
VS = np.array([litres(h) for h in HS])

def barrel(height_m=0.6):
    h = height_m
    d = across(h)
    v = litres(h)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 5), gridspec_kw={"width_ratios": [1, 1.5]})
    # the barrel, standing, seen from the side
    ax.fill([0, d, d, 0], [0, 0, h, h], color=LILAC, alpha=0.18)
    ax.plot([0, d, d, 0, 0], [0, 0, h, h, 0], color=CHALK, lw=2)
    ax.plot([0, d], [h / 2, h], color=CORAL, lw=3.5, label="the rod: 1 m")
    ax.plot([0], [h / 2], "o", color=CORAL, ms=8)
    ax.set_xlim(-0.15, 1.25); ax.set_ylim(-0.05, 2.05); ax.set_aspect("equal")
    ax.set_xlabel("metres"); ax.legend(loc="upper right")
    ax.set_title(f"{h:.2f} m tall · {d * 100:.0f} cm across")
    # the volume against the height
    bx.plot(HS, VS, color=TEAL)
    bx.plot([h], [v], "o", color=CORAL, ms=11)
    bx.axhline(litres(H_BEST), color=DIM, lw=1, ls="--")
    bx.set_xlim(0, 2); bx.set_ylim(0, 700)
    bx.set_xlabel("height (m)"); bx.set_ylabel("litres")
    bx.set_title(f"{v:.0f} litres  ({100 * v / litres(H_BEST):.1f}% of the best)")
    plt.tight_layout(); plt.show()

interact(barrel, height_m=FloatSlider(value=0.6, min=0.1, max=1.9, step=0.05, description="height (m)"));

**Notice.** Drag the height slowly through 1.15 m. Far from the top, each step changes the litres a lot. Near the
top, the dot hardly moves up or down. At 1.2 m, the Austrian shape, it holds 603 litres: 99.8% of the best.

## 2 · Flat at the top

Kepler noticed why: near a maximum, "the decrements on both sides are in the beginning only imperceptible"
(Tikhomirov's translation). Here is the change in litres for one extra centimetre of height, at a few heights.

In [ ]:
print(f"{'height (m)':>11} {'litres':>8} {'one more cm':>13}")
for h in (0.4, 0.8, 1.0, H_BEST, 1.3, 1.6, 1.8):
    print(f"{h:11.3f} {litres(h):8.0f} {litres(h + 0.01) - litres(h):+13.2f}")

**Notice.** The change is positive while the curve rises, negative while it falls, and almost zero at the top. That
change per centimetre is the **slope**, the rate of change of the volume.

## 3 · Fermat's line

About 1636, Pierre de Fermat asked: cut a line 10 long into two pieces. When is the product of the pieces biggest?
He compared the product at x with the product at x + e, a tiny step further, and treated them as nearly equal.

In [ ]:
def fermat(x=3.0):
    xs = np.linspace(0, 10, 300)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
    ax.fill([0, x, x, 0], [0, 0, 10 - x, 10 - x], color=LILAC, alpha=0.25)
    ax.plot([0, x, x, 0, 0], [0, 0, 10 - x, 10 - x, 0], color=CHALK)
    ax.set_xlim(-0.5, 10.5); ax.set_ylim(-0.5, 10.5); ax.set_aspect("equal")
    ax.set_title(f"{x:.1f} × {10 - x:.1f} = {x * (10 - x):.2f}")
    bx.plot(xs, xs * (10 - xs), color=TEAL)
    bx.plot([x], [x * (10 - x)], "o", color=CORAL, ms=11)
    bx.set_xlabel("x, the first piece"); bx.set_ylabel("product"); bx.set_ylim(0, 27)
    bx.set_title("the product x(10 − x)")
    plt.tight_layout(); plt.show()

interact(fermat, x=FloatSlider(value=3, min=0, max=10, step=0.1, description="x"));

**Notice.** At x = 5 the rectangle is a square, and the curve is flat: 4.9 × 5.1 = 24.99, only a hair below 25.

## 4 · Flat doesn't always mean top

The curve f(x) = x³ − 3x has slope 3x² − 3. The slope is zero at x = −1 and at x = 1. Move the point and watch the
slope's sign. The curve y = x³ is flat at 0 too.

In [ ]:
def flat_points(x=-1.8, curve="x³ − 3x"):
    f, df = ((lambda t: t ** 3 - 3 * t), (lambda t: 3 * t ** 2 - 3)) if curve == "x³ − 3x" else \
            ((lambda t: t ** 3), (lambda t: 3 * t ** 2))
    xs = np.linspace(-2.2, 2.2, 300)
    fig, ax = plt.subplots(figsize=(8.5, 5))
    ax.plot(xs, f(xs), color=TEAL)
    s = df(x)
    tx = np.linspace(x - 0.6, x + 0.6, 2)
    ax.plot(tx, f(x) + s * (tx - x), color=SAGE, lw=3)
    ax.plot([x], [f(x)], "o", color=CORAL, ms=10)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-4, 4)
    word = "rising" if s > 1e-9 else ("falling" if s < -1e-9 else "flat")
    ax.set_title(f"y = {curve}:  at x = {x:.1f}, slope {s:+.2f} ({word})")
    plt.show()

interact(flat_points, x=FloatSlider(value=-1.8, min=-2, max=2, step=0.1, description="x"),
         curve=Dropdown(options=["x³ − 3x", "x³"], description="curve"));

**Notice.** On x³ − 3x the slope goes + then − at x = −1 (a hilltop) and − then + at x = 1 (a valley). On x³ it is
+ on both sides of 0: a pause. The **second derivative** (the slope of the slope) tells them apart quickly: for
x³ − 3x it is 6x, which is −6 at −1 (bending down) and +6 at 1 (bending up).

## 5 · Fold a box from any sheet

Cut a square of side x from each corner of a sheet and fold up the sides. The box is x deep, and its base is
(length − 2x) by (width − 2x). Change the sheet, then drag the cut and find the biggest box.

In [ ]:
def box(length_cm=12.0, width_cm=12.0, cut_cm=1.0):
    L, Wd, x = length_cm, width_cm, cut_cm
    xmax = min(L, Wd) / 2
    x = min(x, xmax)
    vol = lambda t: t * (L - 2 * t) * (Wd - 2 * t)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
    ax.plot([0, L, L, 0, 0], [0, 0, Wd, Wd, 0], color=DIM)
    for cx, cy in ((0, 0), (L - x, 0), (0, Wd - x), (L - x, Wd - x)):
        ax.fill([cx, cx + x, cx + x, cx], [cy, cy, cy + x, cy + x], color=CORAL, alpha=0.6)
    ax.plot([x, L - x, L - x, x, x], [x, x, Wd - x, Wd - x, x], color=CHALK, ls="--", lw=1.4)
    ax.set_xlim(-1, max(L, Wd) + 1); ax.set_ylim(-1, max(L, Wd) + 1); ax.set_aspect("equal"); ax.grid(False)
    ax.set_title(f"sheet {L:.0f} × {Wd:.0f} cm · cut {x:.2f} cm")
    ts = np.linspace(0, xmax, 300)
    bx.plot(ts, [vol(t) for t in ts], color=TEAL)
    bx.plot([x], [vol(x)], "o", color=CORAL, ms=11)
    bx.set_xlabel("cut x (cm)"); bx.set_ylabel("volume (cm³)")
    bx.set_title(f"V = {vol(x):.1f} cm³")
    plt.tight_layout(); plt.show()

interact(box, length_cm=FloatSlider(value=12, min=4, max=30, step=1, description="length"),
         width_cm=FloatSlider(value=12, min=4, max=30, step=1, description="width"),
         cut_cm=FloatSlider(value=1, min=0, max=6, step=0.25, description="cut"));

**Notice.** For the 12 cm square, the dot is highest at a cut of 2 cm: 128 cm³. At a cut of 6 cm the base is gone
and the volume is zero. The slope is zero there too, but it is a valley.

## 6 · Check your answer

Calculus finds the top exactly: the slope V′ is zero at the flat points, and the second derivative V″ says whether
each is a hilltop (V″ < 0) or a valley (V″ > 0). The cell below does it with sympy for the 12 cm square.

**Your challenge:** the biggest open box from a card 8 cm by 5 cm. Put your cut (in cm) in `my_cut` and run the
cell. Hint: one of the flat points is too big to cut.

In [ ]:
import sympy as sp
x = sp.symbols("x", positive=True)

V = x * (12 - 2 * x) ** 2
flat = sp.solve(sp.diff(V, x), x)
print("12 cm square:  V′ =", sp.factor(sp.diff(V, x)), "  flat at x =", flat)
for p in flat:
    print(f"   x = {p}: V″ = {sp.diff(V, x, 2).subs(x, p)},  V = {V.subs(x, p)} cm³")

my_cut = None          # ← your answer for the 8 cm × 5 cm card, e.g. my_cut = 1.5

W8 = x * (8 - 2 * x) * (5 - 2 * x)
best = [p for p in sp.solve(sp.diff(W8, x), x) if p < sp.Rational(5, 2) and sp.diff(W8, x, 2).subs(x, p) < 0][0]
if my_cut is None:
    print("\nPut your cut in my_cut and run this cell again.")
elif abs(float(my_cut) - float(best)) < 1e-6:
    print(f"\nRight: a cut of {best} cm gives the biggest box, {W8.subs(x, best)} cm³.")
else:
    print(f"\nNot quite: a cut of {my_cut} cm gives {float(W8.subs(x, my_cut)):.2f} cm³. Look for a bigger box.")

## Sources
Kepler, *Nova Stereometria doliorum vinariorum* (Linz, 1615), Part II, corollary to Theorem V, in V. M. Tikhomirov's
translation (*Stories about Maxima and Minima*, AMS 1990), via Matemáticas Visuales, "Kepler: the volume of a wine
barrel"; T. Christie, "Roll out the barrel" (The Renaissance Mathematicus, 2015); MacTutor, "Pierre de Fermat";
Wikipedia, "Adequality". The barrel is modelled as a cylinder, as Kepler began; real barrels bulge. Every number is
computed in the cells above. All retrieved 2026-10-24.